# Example 10 — Viscous Burgers' equation: the canonical fluid-dynamics PINN

The simplest PDE with real fluid-dynamics structure — **nonlinear convection + viscous
diffusion** — and the benchmark from the original PINN paper (Raissi et al. 2019):
$$u_t + u\,u_x = \nu\,u_{xx},\qquad x\in[-1,1],\ t\in[0,1],\qquad \nu = \tfrac{0.01}{\pi}\approx 0.0032$$
$$u(x,0) = -\sin(\pi x),\qquad u(\pm 1, t) = 0.$$

**The physics:** the nonlinear term $u\,u_x$ self-steepens the wave — fluid moving right
(u>0) catches up with fluid moving left (u<0) at $x=0$, forming a near-shock front whose
thickness is set by the small viscosity. This is the 1-D cartoon of shock formation and of
the convection–diffusion balance at the heart of Navier–Stokes.

**Why it's a good PINN test:** the residual now contains the *nonlinear* term `u*u_x`
(autograd handles it without any linearisation — no Picard/Newton iterations as in CFD),
but the steepening front concentrates high-frequency content at $x=0$ — the spectral-bias
and stiffness themes of Examples 6–7 arrive in a real flow problem.

**Reference solution:** there is no simple closed form, so we do what practitioners do —
compare against a fine-grid **finite-difference solution** (upwind convection + central
diffusion, 2001 points, ~16k steps; runs in under a second).

> Colab: Runtime → GPU. Training takes ~2 min on GPU (~8 min CPU) — the priciest notebook
> in the kit, because the front demands it.

In [ ]:
# Cell 1 -- Reference CFD solution (fine-grid finite differences)
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

NU = 0.01/np.pi

def fd_reference(nx=2001, T=1.0):
    """Upwind convection + central diffusion on a fine grid — our ground truth."""
    x = np.linspace(-1, 1, nx); dx = x[1]-x[0]
    dt = min(0.2*dx**2/NU, 0.4*dx)          # diffusive + convective stability
    nt = int(T/dt)+1; dt = T/nt
    u = -np.sin(np.pi*x)
    hist = {0.0: u.copy()}
    for n in range(nt):
        up = np.maximum(u, 0); um = np.minimum(u, 0)
        dm = np.zeros_like(u); dp = np.zeros_like(u)
        dm[1:]  = (u[1:]-u[:-1])/dx          # backward difference (for u>0)
        dp[:-1] = (u[1:]-u[:-1])/dx          # forward difference  (for u<0)
        diff = np.zeros_like(u)
        diff[1:-1] = NU*(u[2:]-2*u[1:-1]+u[:-2])/dx**2
        u = u + dt*(diff - (up*dm + um*dp))
        u[0] = 0; u[-1] = 0
        for ts in (0.25, 0.5, 0.75, 1.0):
            if abs((n+1)*dt - ts) < dt/2 and ts not in hist: hist[ts] = u.copy()
    return x, hist, nt

t0 = time.perf_counter()
xr, u_ref, nt = fd_reference()
print(f'FD reference: {time.perf_counter()-t0:.1f} s ({nt} steps). '
      f'Watch the front steepen at x=0:')
plt.figure(figsize=(9, 4))
for ts, u in u_ref.items():
    plt.plot(xr, u, lw=1.5, label=f't = {ts}')
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.grid(alpha=.3)
plt.title('Burgers: self-steepening into a viscous near-shock at x = 0')
plt.tight_layout(); plt.show()

In [ ]:
# Cell 2 -- PINN: the nonlinear residual is one line; the front costs epochs
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 64), nn.Tanh(),
                      nn.Linear(64, 64), nn.Tanh(),
                      nn.Linear(64, 64), nn.Tanh(),
                      nn.Linear(64, 64), nn.Tanh(),
                      nn.Linear(64, 1)).to(device)

EPOCHS = 12000
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

x_ic = torch.linspace(-1, 1, 400, device=device).reshape(-1, 1)
u_ic = -torch.sin(np.pi*x_ic)
t_bc = torch.rand(200, 1, device=device)
x_bc = torch.cat([-torch.ones(200, 1, device=device), torch.ones(200, 1, device=device)])
t_bc2 = torch.cat([t_bc, t_bc])

t0 = time.perf_counter()
for e in range(EPOCHS):
    # staged LR decay: constant, then anneal — stabilises the front late in training
    if e == 8000:
        for g in opt.param_groups: g['lr'] = 2e-4
    if e == 10000:
        for g in opt.param_groups: g['lr'] = 5e-5
    opt.zero_grad()
    x = (torch.rand(2500, 1, device=device)*2 - 1).requires_grad_(True)
    t = (torch.rand(2500, 1, device=device)).requires_grad_(True)
    u  = model(torch.cat([x, t], 1))
    ux = torch.autograd.grad(u,  x, torch.ones_like(u),  create_graph=True)[0]
    ut = torch.autograd.grad(u,  t, torch.ones_like(u),  create_graph=True)[0]
    uxx= torch.autograd.grad(ux, x, torch.ones_like(ux), create_graph=True)[0]
    res = ut + u*ux - NU*uxx                 # NONLINEAR term: autograd doesn't care
    loss = (res**2).mean() \
         + 20*((model(torch.cat([x_ic, torch.zeros_like(x_ic)], 1)) - u_ic)**2).mean() \
         + 20*(model(torch.cat([x_bc, t_bc2], 1))**2).mean()
    loss.backward(); opt.step()
    if e % 2000 == 0: print(f'epoch {e:5d}  loss {loss.item():.2e}')
if device.type == 'cuda': torch.cuda.synchronize()
print(f'\nPINN training: {time.perf_counter()-t0:.0f} s')

In [ ]:
# Cell 3 -- PINN vs CFD reference, and the front close-up
xe = torch.tensor(xr, dtype=torch.float32, device=device).reshape(-1, 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))

for ts, uref in u_ref.items():
    with torch.no_grad():
        up = model(torch.cat([xe, torch.full_like(xe, ts)], 1)).cpu().numpy().ravel()
    p = ax[0].plot(xr, uref, lw=2.0, alpha=.5)
    ax[0].plot(xr, up, '--', color=p[0].get_color(), lw=1.2, label=f't = {ts}')
ax[0].set_title('solid = FD reference, dashed = PINN')
ax[0].set_xlabel('x'); ax[0].set_ylabel('u'); ax[0].legend(fontsize=9); ax[0].grid(alpha=.3)

m = np.abs(xr) < 0.2
with torch.no_grad():
    up1 = model(torch.cat([xe, torch.ones_like(xe)], 1)).cpu().numpy().ravel()
err = np.sqrt(np.mean((up1 - u_ref[1.0])**2)/np.mean(u_ref[1.0]**2))
ax[1].plot(xr[m], u_ref[1.0][m], 'g', lw=2.2, label='FD reference')
ax[1].plot(xr[m], up1[m], 'r--', lw=1.6, label=f'PINN (rel L2 = {err:.3f})')
ax[1].set_title('Close-up of the viscous front at t = 1')
ax[1].set_xlabel('x'); ax[1].set_ylabel('u'); ax[1].legend(fontsize=9); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f'relative L2 error at t=1: {err:.4f}')

## Observations (for fluid-dynamics notes)

- **Nonlinearity is free for PINNs.** The residual `ut + u*ux - NU*uxx` handles the
  nonlinear convection term with no linearisation, no Picard/Newton iterations, no upwinding
  decisions — autograd differentiates through the product. That machinery *is* required in
  the FD reference (note its upwind switching on the sign of u).
- **The front is where the budget goes.** Away from $x=0$ the solution is smooth and
  converges in ~2k epochs; the last factor of 10 in error is spent sharpening the viscous
  front (thickness ~$\nu$) — the spectral-bias story of Example 6 in a real flow. Expect
  ~1–2% relative L2 with this Adam-only budget; the literature reaches <0.1% by polishing
  with L-BFGS.
- **Staged LR decay matters.** With a constant learning rate the front position jitters and
  the error bounces between 1% and 15% late in training; annealing freezes it.
- **CFD is still ~1000× faster here** (0.3 s vs minutes) — consistent with Example 1's
  lesson. Burgers is the PINN benchmark not because PINNs beat CFD on it, but because it is
  the smallest problem with Navier–Stokes-like structure.
- **Sanity anchor:** as $\nu \to 0$ this becomes inviscid Burgers with a true shock — the
  strong-form residual becomes undefined at the jump, and shock-capturing tricks (learnable
  artificial viscosity, as in underPINN's Euler cases) become necessary.

**Experiments to try:** raise `NU` to `0.1/np.pi` (mild front — watch training get easy);
try RBA from Example 7 on the collocation points (the weights should pile up along the
front trajectory x=0); drop the IC weight to 1 and watch the solution drift.